# CV II — 転移学習・ファインチューニング

事前学習済みモデル（timm）のファインチューニング、データ拡張、TTA を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/deep-learning/05-transfer-learning`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# timm の確認・自動インストールとクラス数の設定を行います
import subprocess
import sys

try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import torch
n_classes = 5

## 転移学習が主流

画像コンペでは、ImageNet 等の大規模データで事前学習済みのモデルを再利用する
**転移学習**が標準です。少ないデータでも高精度を出せ、学習も速くなります。

`timm`（PyTorch Image Models）を使うと、最新の事前学習済みモデルを 1 行で呼べます。

In [ ]:
import timm
import torch.nn as nn

model = timm.create_model("resnet18", pretrained=True, num_classes=n_classes)

## ファインチューニングの進め方

- **全層を微調整**：データが十分にあるときの基本
- **ヘッドだけ学習 → 段階的に解凍**：データが少ないときに過学習を抑える
- 事前学習時より**小さい学習率**を使う（壊さないため）。層ごとに学習率を変える手法もある

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

In [ ]:
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 3, 224, 224))
print("Pretrained ResNet18 output shape:", out.shape, "| optimizer:", optimizer.__class__.__name__)

## データ拡張と TTA

- **データ拡張**：学習時に反転・回転・切り抜き・色変換などでデータを水増しし過学習を抑える
- **TTA（Test Time Augmentation）**：推論時にも拡張した複数版を予測して平均し、精度を底上げする

> **Tip:** モデルを大きくするより、**適切な拡張 + 転移学習 + 交差検証**の方が安定して効くことが多いです。 まず軽量モデルで CV を固め、その後に大型モデルや TTA を足しましょう。

> **Warning:** 検証時のデータ拡張は推論と整合させ、正規化の統計量は学習時と同じものを使います。 ここがずれると CV が当てにならなくなります。

## 次のステップ

次は NLP の基礎、テキスト前処理と古典的手法（BoW/TF-IDF）を学びます。